# **imports**

In [ ]:
!nvidia-smi

In [ ]:
%pip install torchmetrics

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split, Dataset

import torchvision
from torchvision import datasets
from torchvision import transforms as T
from torchvision.models import vit_b_16 as ViT
from torchvision.models import resnet34 as ResNet34
from torch.utils.data import Dataset
from torchvision.datasets.utils import download_url

import matplotlib.pyplot as plt
from tqdm import tqdm
from torchmetrics import Accuracy
import os
from PIL import Image

import json

# **dataset**

In [ ]:
def expand_if_grayscale(x):
    # 如果只有1個channel才expand
    if x.shape[0] == 1:
        return x.expand(3, -1, -1)
    return x

In [ ]:
transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Lambda(expand_if_grayscale),
    T.Normalize([0.1307], [0.3081])
])
train_data = datasets.EMNIST(root="C:\\Users\\User\\Desktop\\EMNIST\\raw\\train-images-idx3-ubyte", split='balanced', train=True, transform=transforms, download=True)
test_data = datasets.EMNIST(root="C:\\Users\\User\\Desktop\\EMNIST\\raw\\test-images-idx3-ubyte", split='balanced', train=False, transform=transforms, download=True)
class_number = 47

In [ ]:
transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Lambda(expand_if_grayscale),
    T.Normalize([0.1307, 0.1307, 0.1307], [0.3081, 0.3081, 0.3081])
])
train_data = datasets.CIFAR100(root='CIFAR100', train=True, transform=transforms, download=True)
test_data = datasets.CIFAR100(root='CIFAR100', train=False, transform=transforms, download=True)
class_number = 100

In [ ]:
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

with open(r"C:\Users\User\Downloads\archive\Labels.json", 'r') as f:
    idx2label = json.load(f)

train_data = datasets.ImageFolder(root=r"C:\Users\User\Downloads\archive\train.X", transform=transform)
test_data = datasets.ImageFolder(root=r"C:\Users\User\Downloads\archive\val.X", transform=transform)
class_number = len(train_data.classes)

In [ ]:
train_data.class_to_idx

In [ ]:
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [ ]:
# 取出一個 batch，並取得第一個資料
x, y = next(iter(train_loader))
plt.imshow(x[0].permute(1, 2, 0))

# **ResNet Model**

In [ ]:
model = ResNet34(weights=None)
model.fc = nn.Linear(512, class_number)
model.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.layer4[0]

# **ViT model**

In [ ]:
model = ViT(weights=None)
model.heads.head = nn.Linear(model.heads.head.in_features, class_number)

# **Optimizer**

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, nesterov=True, weight_decay=1e-4)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

In [ ]:
loss_fn = nn.CrossEntropyLoss()

# **Train**

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

class AverageMeter(object) :
  def __init__(self):
    self.reset()
  def reset(self) :
    self.avg = 0
    self.val = 0
    self.sum = 0
    self.count = 0
  def update (self, val, n=1) :
    self.val = val
    self.count += n
    self.sum += self.val * n
    self.avg = self.sum / self.count

def train(model, train_loader, optimizer, loss_fn, epoch=None):
  model.train()
  trian_loss = AverageMeter()
  train_acc = Accuracy(task='multiclass', num_classes=class_number).to(device)
  with tqdm(train_loader, unit='batch') as tepoch :
    for inputs, targets in tepoch:
      if epoch is not None:
        tepoch.set_description(f'Epoch {epoch}')
      inputs = inputs.to(device)
      targets = targets.to(device)

      outputs = model(inputs)
      loss = loss_fn(outputs, targets)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

      trian_loss.update(loss.item())
      train_acc(outputs.argmax(dim=1), targets)

      tepoch.set_postfix(loss=trian_loss.avg,
                         accuracy=100*train_acc.compute().item())
    return model, trian_loss.avg, train_acc.compute()

def evaluate (model, test_loader, loss_fn):
  model.eval()
  with torch.no_grad():
    test_loss = AverageMeter()
    test_acc = Accuracy(task='multiclass', num_classes=class_number).to(device)
    for n, (inputs, targets) in enumerate(test_loader):
      inputs =  inputs.to(device)
      targets = targets.to(device)

      outputs = model(inputs)
      loss = loss_fn(outputs, targets)

      test_loss.update(loss)
      test_acc(outputs, targets)
      return test_loss.avg, test_acc.compute()
    

loss_train_hist = []
loss_valid_hist = []

acc_train_hist = []
acc_valid_hist = []

In [ ]:
epoch_number = 15

for epoch in range(epoch_number):
  model, train_loss, train_acc = train(model, train_loader, optimizer, loss_fn, epoch)

  test_loss, test_acc = evaluate(model, test_loader, loss_fn)
  loss_train_hist.append(train_loss)
  loss_valid_hist.append(test_loss.to('cpu'))

  acc_train_hist.append(train_acc.to('cpu'))
  acc_valid_hist.append(test_acc.to('cpu'))
  print(f'Test - Loss:{test_loss} - Accuracy:{test_acc}')
  print()

# **Plot**

In [ ]:
avg_epoch = 15
plt.plot(range(avg_epoch), loss_train_hist, 'k-', label="Train",)
plt.plot(range(avg_epoch), loss_valid_hist, 'y-', label="Validation")

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

In [ ]:
plt.plot(range(avg_epoch), acc_train_hist, 'k-', label='Train')
plt.plot(range(avg_epoch), acc_valid_hist, 'm-', label='Validation',)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.legend()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
torch.save(model, 'C:\\Users\\User\\Documents\\GitHub\\Models-Build\\ResNet-EMNIST-batchsize64-opt=SGD+lr=0.001.pth')